# **ANÁLISIS DE SENTIMIENTO DE REVIEWS**

- Este notebook aplica modelos de procesamiento de lenguaje natural (NLP) para analizar el sentimiento de las reviews de usuarios de aplicaciones de citas.
- El **objetivo** es transformar el contenido textual en variables cuantificables que permitan medir la percepción del usuario, evaluar la experiencia general y facilitar su análisis posterior mediante visualización y modelado.
- Se utilizan modelos preentrenados de Hugging Face, optimizados para tareas de clasificación de sentimiento, que permiten identificar automáticamente el tono de cada review y estimar su nivel de confianza.
- Este proceso convierte datos no estructurados en información analizable, ampliando significativamente el valor analítico del dataset.

#### **0. CONFIGURACIÓN DEL ENTORNO DE ANÁLISIS**

En esta sección se importan las librerías necesarias para el análisis de sentimiento y se configura el entorno de trabajo.

- `pandas`: utilizado para la manipulación, transformación y análisis del dataset.
- `transformers`: proporciona acceso a modelos preentrenados de Hugging Face, que permiten aplicar técnicas avanzadas de procesamiento de lenguaje natural (NLP), incluyendo clasificación de sentimiento.
- `tqdm`: permite monitorizar el progreso de operaciones iterativas, especialmente útil al procesar grandes volúmenes de texto.
- Se ajusta la configuración de visualización de pandas para mostrar el contenido completo de las reviews, facilitando la inspección y validación de los resultados.

Esta configuración inicial establece el entorno necesario para aplicar modelos de análisis de sentimiento de forma eficiente y reproducible.

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', None)

c:\Users\arant\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **1. CARGA DEL DATASET LIMPIO**

- En esta sección se carga el dataset previamente depurado durante la fase de limpieza.
- Este dataset contiene reviews textuales que cumplen los criterios mínimos de calidad semántica y consistencia estructural.
- La carga desde `data/processed/` garantiza que el análisis se realiza sobre una versión validada y preparada para su procesamiento NLP.

In [2]:
df = pd.read_csv('../data/processed/dating_app_reviews_clean.csv')

#### **2. CONFIGURACIÓN DEL MODELO DE ANÁLISIS DE SENTIMIENTO**

- En esta sección se inicializa el modelo de análisis de sentimiento utilizando la librería Hugging Face Transformers.
- Se emplea un modelo preentrenado capaz de clasificar texto según su polaridad emocional (positivo, negativo o neutro).
- Estos modelos han sido entrenados sobre grandes volúmenes de texto y permiten obtener resultados fiables sin necesidad de entrenamiento adicional.
- Esta aproximación optimiza el tiempo de desarrollo y permite aplicar técnicas avanzadas de NLP de forma eficiente.

In [3]:
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    device=-1
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 270.77it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### **3. APLICACIÓN DEL MODELO AL CORPUS DE REVIEWS Y GENERACIÓN DE VARIABLES DE SENTIMIENTO**

- En esta sección se aplica el modelo de sentimiento a cada review del dataset.
- El modelo analiza el contenido textual y devuelve el Dataframe junto con nuevas variables derivadas del análisis de sentimiento:

  - `sentiment`: clasificación del sentimiento
  - `sentiment_score`: nivel de confianza de la predicción

- Este proceso transforma el contenido textual en variables estructuradas que pueden ser utilizadas en análisis cuantitativos posteriores.
- Estas nuevas variables permiten analizar cuantitativamente la percepción del usuario y evaluar patrones de satisfacción y experiencia.

In [ ]:
# preparar textos
texts = df["review"].fillna("").astype(str).tolist()

# función batch para análisis de sentimientos
def analizar_sentimientos_batch(texts, batch_size=32):

    resultados = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Analizando sentimiento"):
        
        batch = texts[i:i+batch_size]
        
        try:
            output = sentiment_analyzer(batch)
            resultados.extend(output)
            
        except Exception:
            resultados.extend(
                [{"label": "ERROR", "score": None}] * len(batch)
            )
    
    return resultados


# ejecutar análisis
results = analizar_sentimientos_batch(texts, batch_size=32)

# añadir columnas al dataframe
df["sentiment"] = [r["label"] for r in results]
df["sent_score"] = [r["score"] for r in results]

# opcional: variable numérica
df["sentiment_map"] = df["sentiment"].map({
    "NEGATIVE": -1,
    "NEUTRAL": 0,
    "POSITIVE": 1
})

# resultado final
df.head()

Analizando sentimiento: 100%|██████████| 1563/1563 [1:44:52<00:00,  4.03s/it] 


,name,review,rating,thumbs_up,date_time,app,year,month,month_name,day,weekday,hour,date,sentiment,sent_score,sentiment_map
0,Diane Berrios,"Ghost Central- It Seems Like There Is A Bunch Of Nice Guys Are There But, It Also Seems Like There'S A Lot Of Ghosting Going Around. The Sadie'S Hawkin Idea Is Really Coolif Guys Actually Responded Or If Guys Actually Are Capable Of Having A Conversation On Here. It'S Always Tough Putting Yourself Out There Epecially When The Guys Like You. It'S Just Not Worth Paying For An App Where Ghosting Is The Norm. This Queen Bee Is Out! Buzz Buzz🐝",2,1,2019-11-22 18:26:00,Bumble,2019,11,November,22,Friday,18,2019-11-22,neutral,0.527266,NaN
1,Universal Solvent,Jhantu App For 90% Indians....Suitable Only For Metro Cities..,1,0,2021-04-14 11:28:00,Tinder,2021,4,April,14,Wednesday,11,2021-04-14,neutral,0.682565,NaN
2,Heather Anthony,"Will Not Load Potential Matches. I Have Been Notified That I Have Been ""Superliked"" But It Still Just Stays On ""Finding People Near You."" It Has Been About A Week, I'Ve Tried Deleting Account And Uninstalling It With No Luck.",2,2,2017-01-10 09:32:00,Tinder,2017,1,January,10,Tuesday,9,2017-01-10,negative,0.756307,NaN
3,Jaylyn Jones,Tinder Loops Not Uploading. Wont Allow Me To Add Loop,3,0,2018-11-05 06:47:00,Tinder,2018,11,November,5,Monday,6,2018-11-05,negative,0.873517,NaN
4,Jayant Vats,Hi Team Hinge I Am Facing With My Matches That Firstly I'M Not Getting Any Notifications For New Matches And Also When I Try To Open Any Match And Try To Text Them It'S Just Showing Loading Symbol And Its Stuck There I'M Not Able To Send Messages Or Anything. Earlier There Was No Such Issue. I Tried Reinstalling It But Still It Doesn'T Work. Can You Please Help Me With This Issue As It'S Getting Irritating. Thank You,2,57,2022-02-02 13:50:00,Hinge,2022,2,February,2,Wednesday,13,2022-02-02,negative,0.783101,NaN


#### **4. DEPURACIÓN Y CONSOLIDACIÓN DE LAS VARIABLES DE SENTIMIENTO**

En esta sección se realiza la depuración final de las variables generadas durante el análisis de sentimiento, con el objetivo de garantizar la calidad y consistencia del dataset.

- Se eliminan las filas cuya clasificación de sentimiento ha producido un error (`sentiment == 'ERROR'`), evitando la incorporación de registros no válidos en el análisis.
- Se elimina la columna auxiliar `sentiment_map`, utilizada únicamente como soporte intermedio durante el proceso de transformación, y que no aporta valor analítico en la versión final del dataset.

Este proceso asegura que el dataset resultante contenga únicamente observaciones válidas y variables relevantes, dejándolo preparado para su uso en fases posteriores de análisis y visualización.

In [ ]:
#eliminar filas con error
df = df[df['sentiment'] != 'ERROR']
#eliminar columna de sentiment_map
df = df.drop(columns=['sentiment_map'])

In [25]:
df.head()


,name,review,rating,thumbs_up,date_time,app,year,month,month_name,day,weekday,hour,date,sentiment,sent_score
0,Diane Berrios,"Ghost Central- It Seems Like There Is A Bunch Of Nice Guys Are There But, It Also Seems Like There'S A Lot Of Ghosting Going Around. The Sadie'S Hawkin Idea Is Really Coolif Guys Actually Responded Or If Guys Actually Are Capable Of Having A Conversation On Here. It'S Always Tough Putting Yourself Out There Epecially When The Guys Like You. It'S Just Not Worth Paying For An App Where Ghosting Is The Norm. This Queen Bee Is Out! Buzz Buzz🐝",2,1,2019-11-22 18:26:00,Bumble,2019,11,November,22,Friday,18,2019-11-22,neutral,0.527266
1,Universal Solvent,Jhantu App For 90% Indians....Suitable Only For Metro Cities..,1,0,2021-04-14 11:28:00,Tinder,2021,4,April,14,Wednesday,11,2021-04-14,neutral,0.682565
2,Heather Anthony,"Will Not Load Potential Matches. I Have Been Notified That I Have Been ""Superliked"" But It Still Just Stays On ""Finding People Near You."" It Has Been About A Week, I'Ve Tried Deleting Account And Uninstalling It With No Luck.",2,2,2017-01-10 09:32:00,Tinder,2017,1,January,10,Tuesday,9,2017-01-10,negative,0.756307
3,Jaylyn Jones,Tinder Loops Not Uploading. Wont Allow Me To Add Loop,3,0,2018-11-05 06:47:00,Tinder,2018,11,November,5,Monday,6,2018-11-05,negative,0.873517
4,Jayant Vats,Hi Team Hinge I Am Facing With My Matches That Firstly I'M Not Getting Any Notifications For New Matches And Also When I Try To Open Any Match And Try To Text Them It'S Just Showing Loading Symbol And Its Stuck There I'M Not Able To Send Messages Or Anything. Earlier There Was No Such Issue. I Tried Reinstalling It But Still It Doesn'T Work. Can You Please Help Me With This Issue As It'S Getting Irritating. Thank You,2,57,2022-02-02 13:50:00,Hinge,2022,2,February,2,Wednesday,13,2022-02-02,negative,0.783101


#### **5. EXPORTACIÓN DEL DATASET ENRIQUECIDO**

- En esta sección se exporta el dataset tras incorporar las variables de sentimiento.
- El dataset resultante se guarda en `data/processed/`, permitiendo su uso en fases posteriores de análisis y visualización.
- Esta exportación evita la necesidad de repetir el procesamiento NLP y asegura la reproducibilidad del pipeline analítico.

In [26]:
df.to_csv('../data/processed/dating_app_reviews_sentiment.csv', index=False)